# Cohort 6: Features Build

In [1]:
%%configure -f
{
    "conf": {
        "spark.broadcast.compress": "true", 
        "spark.jars.packages": "ai.catboost:catboost-spark_3.5_2.12:1.2.7",
        "spark.jars.packages.resolve.transitive": "true",
        "spark.executor.memory": "84g",
        "spark.executor.memoryOverhead": "24g",
        "spark.executor.cores": "1",   
        "spark.executorEnv.CATBOOST_WORKER_INIT_TIMEOUT": "3600s",
        "spark.executor.memoryOverheadFactor": "0.4",
        "spark.executor.extraJavaOptions": "-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35 -XX:ConcGCThreads=2 --add-exports java.base/sun.net.util=ALL-UNNAMED",
        "spark.shuffle.file.buffer": "1m",
        "spark.reducer.maxSizeInFlight": "96m",
        "spark.driver.extraJavaOptions": "--add-exports java.base/sun.net.util=ALL-UNNAMED",
        "spark.driver.memory": "84g",
        "spark.driver.memoryOverhead": "24g",
        "spark.dynamicAllocation.enabled": "true",
        "spark.dynamicAllocation.minExecutors": "4",
        "spark.dynamicAllocation.maxExecutors": "120",
        "spark.memory.fraction": "0.6",
        "spark.memory.storageFraction": "0.4",
        "spark.network.timeout": "1200s",  
        "spark.rpc.askTimeout": "1200s", 
        "spark.rpc.message.maxSize": "512",
        "spark.shuffle.service.enabled": "true",
        "spark.sql.shuffle.partitions": "1000",
        "spark.sql.adaptive.enabled": "true", 
        "spark.sql.broadcastTimeout": "1200s",
        "spark.sql.session.timeout": "1200s",
        "spark.task.cpus": "1",  
        "spark.yarn.am.memory": "12g"
    }
}

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.linalg import Vectors
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
import catboost_spark

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
1,application_1737291559415_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
# Adding a parameter tag
cohort = 'cohort6'

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# S3 Paths
s3_bucket = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/2_enhanced_datasets/{cohort}"
train_input_path = f"{s3_bucket}/train"
test_input_path = f"{s3_bucket}/test"

# Read processed train and test datasets from S3
print("Reading train and test datasets...")
train_df = spark.read.parquet(train_input_path)
test_df = spark.read.parquet(test_input_path)

print("Train and test datasets successfully loaded.")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Reading train and test datasets...
Train and test datasets successfully loaded.

In [5]:
# Verify output
print("Train Dataframe Schema:")
train_df.printSchema()
print("Test Dataframe Schema:")
test_df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Train Dataframe Schema:
root
 |-- mi_person_key: string (nullable = true)
 |-- member_age_dos: integer (nullable = true)
 |-- drug_date: date (nullable = true)
 |-- ADE_Date: date (nullable = true)
 |-- standardized_drug_name: string (nullable = true)
 |-- label: integer (nullable = true)
 |-- person_key_index: double (nullable = true)
 |-- drug_name_index: double (nullable = true)
 |-- drug_name_one_hot: vector (nullable = true)
 |-- features: vector (nullable = true)
 |-- polypharmacy: long (nullable = true)
 |-- activity_count: long (nullable = true)
 |-- polypharmacy_bin: string (nullable = true)
 |-- activity_count_bin: string (nullable = true)
 |-- activity_tag: string (nullable = true)
 |-- partition_key: integer (nullable = true)

Test Dataframe Schema:
root
 |-- mi_person_key: string (nullable = true)
 |-- member_age_dos: integer (nullable = true)
 |-- drug_date: date (nullable = true)
 |-- ADE_Date: date (nullable = true)
 |-- standardized_drug_name: string (nullable = true)


In [6]:
# Drop the 'features' column from the training DataFrame
train_df = train_df.drop("features")

# Drop the 'features' column from the testing DataFrame
test_df = test_df.drop("features")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Train Datasets

In [7]:
# Filter by polypharmacy ranges
low_polypharmacy_train = train_df.filter(col("polypharmacy") <= 3)
moderate_polypharmacy_train = train_df.filter((col("polypharmacy") > 3) & (col("polypharmacy") <= 7))
high_polypharmacy_train = train_df.filter(col("polypharmacy") > 7)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Test Datasets

In [8]:
# Filter by polypharmacy ranges
low_polypharmacy_test = test_df.filter(col("polypharmacy") <= 3)
moderate_polypharmacy_test = test_df.filter((col("polypharmacy") > 3) & (col("polypharmacy") <= 7))
high_polypharmacy_test = test_df.filter(col("polypharmacy") > 7)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

# Cohort 6 Models: Low Polypharmacy

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline

# Add polypharmacy
assembler = VectorAssembler(
    inputCols=["person_key_index", "drug_name_one_hot"],  # Include mi_person_key as an indexed categorical feature
    outputCol="features"
)

# Create pipeline
pipeline = Pipeline(stages=[assembler])

# Fit and transform the data
pipeline_model = pipeline.fit(train_df)

# Transform the dataset
train_low = pipeline_model.transform(low_polypharmacy_train)
test_low = pipeline_model.transform(low_polypharmacy_test)

In [ ]:
# CatBoost Pool objects
from pyspark import StorageLevel

# Cache or persist the Spark DataFrames before creating the Pool
train_df = train_low.select("member_age_dos", "features", "label").persist(StorageLevel.MEMORY_AND_DISK)
test_df = test_low.select("member_age_dos", "features", "label").persist(StorageLevel.MEMORY_AND_DISK)

# Create the CatBoost Pool objects
train_pool = catboost_spark.Pool(train_df)
test_pool = catboost_spark.Pool(test_df)

# Confirm the DataFrames are cached/persisted
print(train_df.storageLevel)
print(test_df.storageLevel)

In [ ]:
# Seeds for different runs - 10 models
seeds = [3, 24, 18, 17, 19, 11, 38, 74, 35, 90]

# Start model number tracker
model_num = 1

# Loop to train and save models (10 runs for stable feature selection)
for seed in seeds:
    print(f"Training model {model_num} with seed {seed}...")
    
    # Initialize CatBoost Spark Classifier with the current seed
    classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

    # Train the model
    model = classifier.fit(train_pool, evalDatasets=[test_pool])

    # Define the path to save the Spark model, including the model number
    spark_model_path = f"s3://pgx-repository/ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}/spark_model_{model_num}"

    # Save the Spark model (with metadata)
    model.write().overwrite().save(spark_model_path)

    print(f"Spark model {model_num} with metadata saved to: {spark_model_path}")
    
    # Clean up memory for next run
    del classifier
    del model
    
    # Increment the model number for the next run
    model_num += 1

# Feature Subsetting to Reduce Memory Requirements

In [10]:
import boto3
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

# Function to list already saved models in the S3 bucket
def get_existing_models(bucket_name, prefix):
    s3 = boto3.client('s3')
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    existing_models = set()
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                # Remove the prefix and split remaining path
                relative_path = key[len(prefix):].lstrip('/')
                parts = relative_path.split('/')
                if len(parts) > 0:
                    # Only add the top-level folder name (model identifier)
                    model_identifier = parts[0]
                    existing_models.add(model_identifier)
    return existing_models


# S3 bucket and prefix
bucket_name = "pgx-repository"
prefix = f"ade-risk-model/Step5_Time_to_Event_Model/4_models/{cohort}"

# Fetch existing models from S3
existing_models = get_existing_models(bucket_name, prefix)

# Define a UDF to get the size of the sparse vector
def get_vector_size(sparse_vector):
    return sparse_vector.size

vector_size_udf = udf(get_vector_size, IntegerType())

print(existing_models)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

{'spark_model_1009_subset_1', 'spark_model_1017_subset_2', 'spark_model_5', 'spark_model_1005_subset_1', 'spark_model_1019_subset_2', 'spark_model_1033_subset_4', 'spark_model_1004_subset_1', 'spark_model_1028_subset_3', 'spark_model_1010_subset_2', 'spark_model_1025_subset_3', 'spark_model_8', 'spark_model_1031_subset_4', 'spark_model_1032_subset_4', 'spark_model_1022_subset_3', 'spark_model_1036_subset_4', 'spark_model_1016_subset_2', 'spark_model_1035_subset_4', 'spark_model_2', 'spark_model_1029_subset_3', 'spark_model_1021_subset_3', 'spark_model_1013_subset_2', 'spark_model_1007_subset_1', 'spark_model_1023_subset_3', 'spark_model_1011_subset_2', 'spark_model_1024_subset_3', 'spark_model_1006_subset_1', 'spark_model_1037_subset_4', 'spark_model_1030_subset_4', 'spark_model_1001_subset_1', 'spark_model_1002_subset_1', 'spark_model_1020_subset_3', 'spark_model_1027_subset_3', 'spark_model_4', 'spark_model_7', 'spark_model_1026_subset_3', 'spark_model_1012_subset_2', 'spark_model_10

# Cohort 6 Models: Moderate Polypharmacy

In [11]:
moderate_polypharmacy_train.schema

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

StructType([StructField('mi_person_key', StringType(), True), StructField('member_age_dos', IntegerType(), True), StructField('drug_date', DateType(), True), StructField('ADE_Date', DateType(), True), StructField('standardized_drug_name', StringType(), True), StructField('label', IntegerType(), True), StructField('person_key_index', DoubleType(), True), StructField('drug_name_index', DoubleType(), True), StructField('drug_name_one_hot', VectorUDT(), True), StructField('polypharmacy', LongType(), True), StructField('activity_count', LongType(), True), StructField('polypharmacy_bin', StringType(), True), StructField('activity_count_bin', StringType(), True), StructField('activity_tag', StringType(), True), StructField('partition_key', IntegerType(), True)])

In [13]:
from pyspark.ml.feature import VectorSlicer
import catboost_spark

# Calculate the size of the vector
vector_size = moderate_polypharmacy_train.select(
    vector_size_udf("drug_name_one_hot").alias("vector_size")
).first()[0]
print(f"Vector size: {vector_size}")

# Determine the number of subsets
subset_size = 5  # Number of features per subset
num_subsets = (vector_size + subset_size - 1) // subset_size  # Ceiling division

print(num_subsets)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Vector size: 7817
1564

In [12]:
# Track all assessed features to avoid duplicates
assessed_indices = set()

# Seeds for different runs - 10 models
seeds = [13, 42, 81, 71, 91, 22, 83, 47, 53, 9]

# Start model number tracker
model_num = 1000

# Loop through subsets of features
for subset_idx in range(num_subsets):
    print(f"Processing subset {subset_idx + 1}/{num_subsets}...")

    # Calculate the indices for the current subset
    start_idx = subset_idx * subset_size
    end_idx = min((subset_idx + 1) * subset_size, vector_size)
    subset_indices = range(start_idx, end_idx)

    # Ensure no duplicate indices
    assert not assessed_indices.intersection(subset_indices), \
        f"Duplicate indices found in subset {subset_idx + 1}!"

    # Track assessed indices
    assessed_indices.update(subset_indices)

    # Train and save models for this subset
    for seed in seeds:
        model_key = f"spark_model_{model_num}_subset_{subset_idx + 1}"
        if model_key in existing_models:
            print(f"Model {model_key} already exists in S3. Skipping...")
            model_num += 1
            continue

        # CatBoost Pools are created only if a model for the current seed needs to be trained
        print(f"Preparing data for subset {subset_idx + 1}...")

        # Create a VectorSlicer for the current subset
        slicer = VectorSlicer(inputCol="drug_name_one_hot", outputCol="features")
        slicer.setIndices(list(subset_indices))

        # Apply the slicer to the training and test DataFrames
        train_df_subset = slicer.transform(moderate_polypharmacy_train).select("features", "label")
        test_df_subset = slicer.transform(moderate_polypharmacy_test).select("features", "label")

        # Create the CatBoost Pool objects for this subset
        train_pool = catboost_spark.Pool(train_df_subset)
        test_pool = catboost_spark.Pool(test_df_subset)

        print(f"Training model {model_num} for feature subset {subset_idx + 1} with seed {seed}...")

        # Initialize CatBoost Spark Classifier with the current seed
        classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

        # Train the model
        model = classifier.fit(train_pool, evalDatasets=[test_pool])

        # Define the path to save the Spark model, including the model number and subset
        spark_model_path = f"s3://{bucket_name}/{prefix}/{model_key}"

        # Save the Spark model (with metadata)
        model.write().overwrite().save(spark_model_path)

        print(f"Spark model {model_num} with metadata for subset {subset_idx + 1} saved to: {spark_model_path}")

        # Clean up memory for the next run
        del classifier
        del model
        del train_pool
        del test_pool

        # Increment the model number for the next run
        model_num += 1

# Verify all features have been assessed without duplicates
assert len(assessed_indices) == vector_size, "Not all features were assessed!"
print("All features have been assessed without duplicates.")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-11:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 178, in cell_monitor
    job_binned_stages[job_id][stage_id] = all_stages[stage_id]
KeyError: 1296
Interrupted by user


# Cohort 6 Models: High Polypharmacy

In [ ]:
high_polypharmacy_train.schema

In [14]:
# Calculate the size of the vector
vector_size = high_polypharmacy_train.select(
    vector_size_udf("drug_name_one_hot").alias("vector_size")
).first()[0]
print(f"Vector size: {vector_size}")

# Determine the number of subsets
subset_size = 5  # Number of features per subset
num_subsets = (vector_size + subset_size - 1) // subset_size  # Ceiling division

print(num_subsets)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Vector size: 7817
1564

In [ ]:
# Track all assessed features to avoid duplicates
assessed_indices = set()

# Seeds for different runs - 10 models
seeds = [13, 42, 81, 71, 91, 22, 83, 47, 53, 9]

# Start model number tracker
model_num = 3000

# Loop through subsets of features
for subset_idx in range(num_subsets):
    print(f"Processing subset {subset_idx + 1}/{num_subsets}...")

    # Calculate the indices for the current subset
    start_idx = subset_idx * subset_size
    end_idx = min((subset_idx + 1) * subset_size, vector_size)
    subset_indices = range(start_idx, end_idx)

    # Ensure no duplicate indices
    assert not assessed_indices.intersection(subset_indices), \
        f"Duplicate indices found in subset {subset_idx + 1}!"

    # Track assessed indices
    assessed_indices.update(subset_indices)

    # Check if all models for this subset are already in S3
    subset_models_exist = all(
        f"spark_model_{model_num + i}_subset_{subset_idx + 1}" in existing_models
        for i in range(len(seeds))
    )
    if subset_models_exist:
        print(f"All models for subset {subset_idx + 1} already exist. Skipping subset...")
        model_num += len(seeds)
        continue

    # Create a VectorSlicer for the current subset
    slicer = VectorSlicer(inputCol="drug_name_one_hot", outputCol="features")
    slicer.setIndices(list(subset_indices))

    # Apply the slicer to the training and test DataFrames
    train_df_subset = slicer.transform(high_polypharmacy_train).select("features", "label")
    test_df_subset = slicer.transform(high_polypharmacy_test).select("features", "label")

    # Train and save 10 models for the current subset
    for seed in seeds:
        model_key = f"spark_model_{model_num}_subset_{subset_idx + 1}"
        if model_key in existing_models:
            print(f"Model {model_key} already exists in S3. Skipping...")
            model_num += 1
            continue

        print(f"Training model {model_num} for feature subset {subset_idx + 1} with seed {seed}...")

        # Initialize CatBoost Spark Classifier with the current seed
        classifier = catboost_spark.CatBoostClassifier(randomSeed=seed)

        # Train the model
        model = classifier.fit(train_pool, evalDatasets=[test_pool])

        # Define the path to save the Spark model, including the model number and subset
        spark_model_path = f"s3://{bucket_name}/{prefix}/{model_key}"

        # Save the Spark model (with metadata)
        model.write().overwrite().save(spark_model_path)

        print(f"Spark model {model_num} with metadata for subset {subset_idx + 1} saved to: {spark_model_path}")

        # Clean up memory for the next run
        del classifier
        del model

        # Increment the model number for the next run
        model_num += 1

# Verify all features have been assessed without duplicates
assert len(assessed_indices) == vector_size, "Not all features were assessed!"
print("All features have been assessed without duplicates.")
